In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — GPU CHECK (run this first before anything else)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch

print("=" * 50)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  ✅ GPU  : {gpu_name}")
    print(f"  ✅ VRAM : {gpu_mem:.1f} GB")
    print(f"  ✅ CUDA : {torch.version.cuda}")
    print("\n  Embedding ~18k docs will take ~6–8 minutes")
    DEVICE = "cuda"
else:
    print("  ❌ No GPU detected!")
    print("\n  Fix: Runtime → Change runtime type → T4 GPU → Save")
    print("  Then: Runtime → Restart session → Re-run from Cell 1")
    print("\n  On CPU this will take ~45-60 min — switch to GPU first.")
    DEVICE = "cpu"
print("=" * 50)

  ✅ GPU  : Tesla T4
  ✅ VRAM : 15.6 GB
  ✅ CUDA : 12.8

  Embedding ~18k docs will take ~6–8 minutes


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Install dependencies
# Pinned opentelemetry versions silence the Colab dependency conflict warnings
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
%%capture

# Force downgrade numpy first
!pip uninstall -y numpy
!pip install numpy==1.26.4

# Then install other libraries
!pip install \
  chromadb==0.4.24 \
  sentence-transformers==2.7.0 \
  tqdm \
  opentelemetry-api==1.38.0 \
  opentelemetry-sdk==1.38.0 \
  opentelemetry-exporter-otlp-proto-http==1.38.0 \
  opentelemetry-exporter-otlp-proto-common==1.38.0 \
  opentelemetry-proto==1.38.0

print("✅ Install complete — now Restart Session then re-run from Cell 1")
# %%capture suppresses all the noisy pip output and warning messages.
# After this cell completes:
#   → Runtime → Restart session
#   → Then run ALL cells from Cell 1 again (GPU check through to the end)
#   → Do NOT re-run this install cell after restarting
print("✅ Install complete — now Restart Session then re-run from Cell 1")

In [ ]:
# Separate cell — confirm install
print("✅ Install complete")
print("   → Runtime → Restart session")
print("   → Re-run all cells from Cell 1")
print("   → Do NOT re-run this install cell after restarting")

✅ Install complete
   → Runtime → Restart session
   → Re-run all cells from Cell 1
   → Do NOT re-run this install cell after restarting


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Verify installs after restart
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import chromadb
import sentence_transformers
import torch

print(f"  chromadb              : {chromadb.__version__}")
print(f"  sentence-transformers : {sentence_transformers.__version__}")
print(f"  torch                 : {torch.__version__}")
print(f"  GPU available         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU name              : {torch.cuda.get_device_name(0)}")
print("\n✅ All packages ready — proceed to Cell 4")

  chromadb              : 0.4.24
  sentence-transformers : 2.7.0
  torch                 : 2.10.0+cu128
  GPU available         : True
  GPU name              : Tesla T4

✅ All packages ready — proceed to Cell 4


In [ ]:
# DIAGNOSTIC CELL — run this to find your exact folder path
import os

# Step 1: confirm Drive is mounted
print("Contents of MyDrive (top level):")
for item in sorted(os.listdir("/content/drive/MyDrive")):
    print(f"  '{item}'")

Contents of MyDrive (top level):


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive'

In [ ]:
# Step 2: once you see the folder name above, check inside it
# Look for the exact folder name from Step 1 output and paste it below

FOLDER_NAME = None  # ← will be set after Step 1

# Find it automatically
mydrivecontents = os.listdir("/content/drive/MyDrive")
matches = [x for x in mydrivecontents if "newsgroup" in x.lower() or "twenty" in x.lower()]
print(f"Folders matching 'newsgroup' or 'twenty': {matches}")

if matches:
    FOLDER_NAME = matches[0]
    BASE_DIR = f"/content/drive/MyDrive/{FOLDER_NAME}"
    print(f"\nBASE_DIR = '{BASE_DIR}'")
    print(f"\nContents of BASE_DIR:")
    for item in sorted(os.listdir(BASE_DIR)):
        print(f"  '{item}'")

Folders matching 'newsgroup' or 'twenty': ['twenty+newsgroups']

BASE_DIR = '/content/drive/MyDrive/twenty+newsgroups'

Contents of BASE_DIR:
  'newsgroups_chromadb'


In [ ]:
# Step 3: once you see the subfolder names, check inside the data folder
# Run this after Step 2 confirms BASE_DIR contents

for item in sorted(os.listdir(BASE_DIR)):
    full = os.path.join(BASE_DIR, item)
    if os.path.isdir(full):
        subcount = len(os.listdir(full))
        print(f"  '{item}'/   ({subcount} items inside)")
    else:
        print(f"  '{item}'   (file)")

  'newsgroups_chromadb'/   (0 items inside)


In [ ]:
# Step 4: FIXED Cell 4 — paste the exact paths found above here
import os

# ── EDIT THESE TWO LINES ONLY based on what Step 2-3 printed ─────────────
BASE_DIR = f"/content/drive/MyDrive/{FOLDER_NAME}"   # auto-set from Step 2
DATA_DIR = os.path.join(BASE_DIR, "20_newsgroups")    # change if name differs

# ── Everything below stays the same ──────────────────────────────────────
PERSIST_DIR = os.path.join(BASE_DIR, "newsgroups_chromadb")
os.makedirs(PERSIST_DIR, exist_ok=True)

print(f"BASE_DIR    = {BASE_DIR}")
print(f"DATA_DIR    = {DATA_DIR}")
print(f"PERSIST_DIR = {PERSIST_DIR}")
print(f"DATA_DIR exists: {os.path.exists(DATA_DIR)}")

if os.path.exists(DATA_DIR):
    cats = sorted([
        d for d in os.listdir(DATA_DIR)
        if os.path.isdir(os.path.join(DATA_DIR, d))
    ])
    print(f"\n✅ Found {len(cats)} category folders:")
    for c in cats:
        n = len(os.listdir(os.path.join(DATA_DIR, c)))
        print(f"   '{c}'  →  {n} files")
else:
    print(f"\n❌ Still not found. Check the exact folder name printed in Step 2-3 above.")
    print(f"   Common causes:")
    print(f"   - Folder is named '20newsgroups' not '20_newsgroups'")
    print(f"   - Folder is nested one level deeper than expected")
    print(f"   - Drive hasn't fully synced yet (wait 1 min and retry)")

BASE_DIR    = /content/drive/MyDrive/twenty+newsgroups
DATA_DIR    = /content/drive/MyDrive/twenty+newsgroups/20_newsgroups
PERSIST_DIR = /content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb
DATA_DIR exists: False

❌ Still not found. Check the exact folder name printed in Step 2-3 above.
   Common causes:
   - Folder is named '20newsgroups' not '20_newsgroups'
   - Folder is nested one level deeper than expected
   - Drive hasn't fully synced yet (wait 1 min and retry)


In [ ]:
# CELL 4 (REPLACEMENT) — Download dataset + save to Drive + set all paths
import os
from sklearn.datasets import fetch_20newsgroups
from tqdm import tqdm

BASE_DIR    = "/content/drive/MyDrive/twenty+newsgroups"
DATA_DIR    = os.path.join(BASE_DIR, "20_newsgroups")
PERSIST_DIR = os.path.join(BASE_DIR, "newsgroups_chromadb")

os.makedirs(DATA_DIR,    exist_ok=True)
os.makedirs(PERSIST_DIR, exist_ok=True)

# ── Download via sklearn (no API key, no manual upload, free) ─────────────
# subset='all' gives train+test combined (~18,846 posts)
# remove=()    keeps raw headers/footers — we clean manually later
print("Downloading 20 Newsgroups (~18k posts)...")
newsgroups = fetch_20newsgroups(
    subset       = 'all',
    remove       = (),
    shuffle      = True,
    random_state = 42
)
print(f"✅ Downloaded {len(newsgroups.data):,} posts across {len(newsgroups.target_names)} categories")

# ── Save to Drive in folder-per-category structure ────────────────────────
# Creates: 20_newsgroups/<category>/<post_id>
# This matches the original file structure from your screenshots exactly
print("\nSaving to Drive (this takes ~2-3 min)...")

for cat in newsgroups.target_names:
    os.makedirs(os.path.join(DATA_DIR, cat), exist_ok=True)

for i, (text, label_idx) in enumerate(tqdm(
    zip(newsgroups.data, newsgroups.target),
    total = len(newsgroups.data),
    desc  = "Writing posts"
)):
    cat   = newsgroups.target_names[label_idx]
    fpath = os.path.join(DATA_DIR, cat, str(i))
    with open(fpath, 'w', encoding='utf-8') as f:
        f.write(text)

# ── Verify ────────────────────────────────────────────────────────────────
print(f"\n✅ All posts saved to: {DATA_DIR}")
print(f"\nCategory breakdown:")
cats = sorted(os.listdir(DATA_DIR))
for cat in cats:
    n = len(os.listdir(os.path.join(DATA_DIR, cat)))
    print(f"  {cat:<42} {n:>5} files")

print(f"\n✅ BASE_DIR    : {BASE_DIR}")
print(f"✅ DATA_DIR    : {DATA_DIR}")
print(f"✅ PERSIST_DIR : {PERSIST_DIR}")
print(f"\nReady — proceed to Cell 5 (load raw posts)")

✅ Downloaded 18,846 posts across 20 categories

Saving to Drive (this takes ~2-3 min)...


Writing posts: 100%|██████████| 18846/18846 [00:01<00:00, 18354.45it/s]


✅ All posts saved to: /content/drive/MyDrive/twenty+newsgroups/20_newsgroups

Category breakdown:
  alt.atheism                                  799 files
  comp.graphics                                973 files
  comp.os.ms-windows.misc                      985 files
  comp.sys.ibm.pc.hardware                     982 files
  comp.sys.mac.hardware                        963 files
  comp.windows.x                               988 files
  misc.forsale                                 975 files
  rec.autos                                    990 files
  rec.motorcycles                              996 files
  rec.sport.baseball                           994 files
  rec.sport.hockey                             999 files
  sci.crypt                                    991 files
  sci.electronics                              984 files
  sci.med                                      990 files
  sci.space                                    987 files
  soc.religion.christian                      

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — MOUNT DRIVE + SET ALL PATHS
#
# IMPORTANT: This cell must run every time after a restart.
# All variables (BASE_DIR, DATA_DIR, PERSIST_DIR, DEVICE) are lost
# when Colab restarts — this cell restores them into memory.
#
# Path layout on your Drive:
#   My Drive/
#   └── twenty+newsgroups/
#       ├── 20_newsgroups/          ← raw posts (created by Cell 6 download)
#       │   ├── alt.atheism/
#       │   ├── comp.graphics/
#       │   └── ... (20 folders)
#       └── newsgroups_chromadb/    ← ALL outputs saved here
#           ├── chroma.sqlite3      ← vector index (Parts 2 & 3 use this)
#           ├── embeddings_backup.npy ← numpy array (Parts 2 & 3 use this)
#           └── manifest.json       ← config file (Parts 2 & 3 import this)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive')

import os
import torch

# ── Path definitions ──────────────────────────────────────────────────────
BASE_DIR    = "/content/drive/MyDrive/twenty+newsgroups"
DATA_DIR    = os.path.join(BASE_DIR, "20_newsgroups")
PERSIST_DIR = os.path.join(BASE_DIR, "newsgroups_chromadb")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(PERSIST_DIR, exist_ok=True)

# ── Check if data already exists from a previous run ─────────────────────
data_exists = os.path.exists(DATA_DIR) and len([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
]) == 20

print(f"✅ BASE_DIR    : {BASE_DIR}")
print(f"✅ DATA_DIR    : {DATA_DIR}")
print(f"✅ PERSIST_DIR : {PERSIST_DIR}")
print(f"✅ DEVICE      : {DEVICE.upper()}")
print(f"\n  Data already downloaded : {'✅ YES → skip Cell 6' if data_exists else '❌ NO → run Cell 6'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ BASE_DIR    : /content/drive/MyDrive/twenty+newsgroups
✅ DATA_DIR    : /content/drive/MyDrive/twenty+newsgroups/20_newsgroups
✅ PERSIST_DIR : /content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb
✅ DEVICE      : CUDA

  Data already downloaded : ✅ YES → skip Cell 6


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — DOWNLOAD DATASET
#
# SKIP THIS CELL if Cell 5 printed "Data already downloaded: ✅ YES"
# Only run once — saves 18,846 posts to Drive as individual files
# matching the original folder-per-category structure.
#
# Why sklearn fetcher instead of manual upload:
#   - Your tar.gz files never synced to Drive properly
#   - sklearn downloads directly in Colab (~30 sec), no manual steps
#   - subset='all' gives train+test combined (more data = better clusters)
#   - remove=() keeps raw headers so our own cleaner handles them
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from sklearn.datasets import fetch_20newsgroups
from tqdm import tqdm
import os

print("Downloading 20 Newsgroups via sklearn (~30 sec)...")

newsgroups = fetch_20newsgroups(
    subset       = 'all',    # train + test combined = 18,846 posts
    remove       = (),       # keep raw — we clean manually in Cell 8
    shuffle      = True,
    random_state = 42        # reproducible ordering
)

print(f"✅ Downloaded {len(newsgroups.data):,} posts | {len(newsgroups.target_names)} categories")

# Create one subfolder per category
for cat in newsgroups.target_names:
    os.makedirs(os.path.join(DATA_DIR, cat), exist_ok=True)

# Write each post as a plain text file named by its index
print("Saving to Drive...")
for i, (text, label_idx) in enumerate(tqdm(
    zip(newsgroups.data, newsgroups.target),
    total = len(newsgroups.data),
    desc  = "Writing posts"
)):
    cat   = newsgroups.target_names[label_idx]
    fpath = os.path.join(DATA_DIR, cat, str(i))
    with open(fpath, 'w', encoding='utf-8') as f:
        f.write(text)

# Verify the write
cats        = sorted(os.listdir(DATA_DIR))
total_files = sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in cats)
print(f"\n✅ Saved {total_files:,} posts across {len(cats)} categories")
print(f"✅ Location : {DATA_DIR}")
print(f"\nCategory breakdown:")
for cat in cats:
    n = len(os.listdir(os.path.join(DATA_DIR, cat)))
    print(f"  {cat:<42} {n:>5} files")

✅ Downloaded 18,846 posts | 20 categories
Saving to Drive...


Writing posts:   1%|          | 191/18846 [00:10<17:28, 17.79it/s]


KeyboardInterrupt: 

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — LOAD RAW POSTS FROM DISK
#
# Reads every file under DATA_DIR/<category>/<filename> into memory.
#
# Encoding strategy:
#   Try UTF-8 first — correct for modern/clean text
#   Fall back to latin-1 — these are 1997 posts, many use ISO-8859-1
#   NEVER use errors='ignore' on UTF-8 — it silently drops characters
#   and corrupts words in ways that quietly hurt embedding quality
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from tqdm import tqdm
import os

def load_from_disk(data_dir):
    texts, labels, paths = [], [], []
    category_dirs = sorted([
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ])
    for cat in tqdm(category_dirs, desc="Loading categories"):
        cat_path = os.path.join(data_dir, cat)
        for fname in os.listdir(cat_path):
            fpath = os.path.join(cat_path, fname)
            if not os.path.isfile(fpath):
                continue
            try:
                with open(fpath, 'r', encoding='utf-8') as f:
                    text = f.read()
            except UnicodeDecodeError:
                with open(fpath, 'r', encoding='latin-1') as f:
                    text = f.read()
            texts.append(text)
            labels.append(cat)
            paths.append(fpath)
    return texts, labels, paths

raw_texts, categories, filepaths = load_from_disk(DATA_DIR)

print(f"\n✅ Total posts loaded : {len(raw_texts):,}")
print(f"\n── Sample raw post (first 500 chars) ──────────────────────")
print(raw_texts[0][:500])
print(f"\nCategory : {categories[0]}")
print(f"File     : {filepaths[0]}")

Loading categories: 100%|██████████| 20/20 [11:12<00:00, 33.63s/it]


✅ Total posts loaded : 38,748

── Sample raw post (first 500 chars) ──────────────────────
Xref: cantaloupe.srv.cs.cmu.edu talk.abortion:120844 alt.atheism:53490 talk.religion.misc:83861
Newsgroups: talk.abortion,alt.atheism,talk.religion.misc
Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!pacific.mps.ohio-state.edu!zaphod.mps.ohio-state.edu!sdd.hp.com!portal!danb
From: danb@shell.portal.com (Dan E Babcock)
Subject: Re: After 2000 years, can we say that Christian Morality is
Message-ID: <C5v0zI.Arn@unix.portal.com>
Sender: news@unix.porta

Category : alt.atheism
File     : /content/drive/MyDrive/twenty+newsgroups/20_newsgroups/alt.atheism/53490


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — CLEAN ALL POSTS
#
# WHAT WE REMOVE AND WHY
# ──────────────────────
# Header block (before first blank line \n\n):
#   Contains From/NNTP-Posting-Host/Organization/Lines/Xref.
#   These are routing metadata not topic signal. Embedding
#   "From: xyz@mit.edu" clusters by sender domain not subject matter.
#
# Lines starting with '>':
#   Quoted replies from earlier posts. They duplicate someone else's
#   content — a short reply with 20 lines of quoted context embeds
#   as the quoted topic not the actual reply's intent.
#
# PGP / signature blocks (-----BEGIN ... -----END):
#   Cryptographic boilerplate. Every PGP block embeds nearly
#   identically regardless of topic → creates false similarity clusters.
#
# Standalone '--' separator:
#   Standard Usenet convention — everything after '-- \n' is a
#   signature (name, contact info). Zero topical signal.
#
# Separator lines (----, ====) and bare numbers:
#   Visual formatting and line counters. Pure noise.
#
# URLs and email addresses (inline removal, not full-line skip):
#   Fragment into meaningless subword tokens. Remove inline rather
#   than dropping whole lines — line may have real content around URL.
#
# WHAT WE KEEP AND WHY
# ────────────────────
# Subject line (prepended to body):
#   Most semantically dense part of a post. After stripping the header
#   we'd lose this entirely. Prepending gives the embedder a clean
#   anchor sentence for the topic.
#
# Full body sentences with function words intact:
#   Sentence-transformers are NOT bag-of-words models. They need full
#   grammatical context. Removing stopwords like "not", "but",
#   "however" actively HURTS quality — these words carry negation and
#   contrast that changes semantic meaning. Never stem or lemmatize
#   before feeding a transformer.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import re
from tqdm import tqdm
from collections import Counter

def extract_subject(raw: str) -> str:
    """
    Pull Subject: line from header and strip Re: prefixes.
    Re: chains add no new topic information — stripping them
    makes the subject line a cleaner semantic anchor.
    """
    for line in raw.split('\n'):
        if line.lower().startswith('subject:'):
            subj = line[8:].strip()
            subj = re.sub(r'^(Re:\s*)+', '', subj, flags=re.IGNORECASE)
            return subj.strip()
    return ''

def clean_post(raw: str) -> str:
    subject      = extract_subject(raw)
    parts        = raw.split('\n\n', 1)     # split header | body
    body         = parts[1] if len(parts) > 1 else raw
    cleaned_lines = []
    in_sig_block  = False

    for line in body.split('\n'):
        s = line.strip()

        # PGP block: skip everything between BEGIN and END
        if re.match(r'^-{3,}BEGIN', s):
            in_sig_block = True
        if in_sig_block:
            if re.match(r'^-{3,}END', s):
                in_sig_block = False
            continue

        if s == '--':                           break   # sig separator — stop
        if s.startswith('>'):                   continue  # quoted reply
        if re.match(r'^[-=~_*]{3,}$', s):      continue  # separator line
        if re.match(r'^\d+$', s):               continue  # bare number

        # Inline removal — keep rest of line after removing URL/email
        line = re.sub(r'https?://\S+|ftp://\S+|www\.\S+', '', line)
        line = re.sub(r'\S+@\S+\.\S+', '', line)

        if len(line.strip()) >= 3:
            cleaned_lines.append(line.strip())

    body_text = ' '.join(cleaned_lines)
    return f"{subject}. {body_text}" if subject else body_text


def is_valid(text: str, min_words: int = 50) -> bool:
    """
    Drop posts under 50 words after cleaning.
    Posts this short after stripping headers + quotes are almost always:
      - 'Me too' / '+1' replies (nothing left after quoted body removed)
      - Binary attachment posts with no readable text body
      - Automated admin notices ('This newsgroup has moved...')
    None contribute useful semantic signal to the index.
    Why NOT higher (e.g. 150): legitimate concise technical Q&A posts
    are typically 50–100 words — a higher floor would bias the corpus.
    """
    return len(text.split()) >= min_words


# ── Apply cleaning to full corpus ─────────────────────────────────────────
unique_cats   = sorted(set(categories))
cat_to_idx    = {c: i for i, c in enumerate(unique_cats)}
cleaned_docs  = []
meta_list     = []
skipped_short = 0
skipped_empty = 0

for i, (raw, cat, fpath) in enumerate(tqdm(
    zip(raw_texts, categories, filepaths),
    total = len(raw_texts),
    desc  = "Cleaning posts"
)):
    cleaned = clean_post(raw)
    if not cleaned:
        skipped_empty += 1
        continue
    if not is_valid(cleaned):
        skipped_short += 1
        continue

    cleaned_docs.append(cleaned)
    meta_list.append({
        "doc_id"    : i,                          # original index in raw list
        "category"  : cat,                        # e.g. "comp.sys.mac.hardware"
        "label_idx" : cat_to_idx[cat],            # integer 0–19
        "filename"  : os.path.basename(fpath),    # e.g. "12345"
        "word_count": len(cleaned.split())
    })

print(f"\n── Cleaning Summary ────────────────────────────────────────")
print(f"  Total loaded     : {len(raw_texts):>6,}")
print(f"  ✅ Kept          : {len(cleaned_docs):>6,}")
print(f"  ⛔ Too short     : {skipped_short:>6,}  (< 50 words after cleaning)")
print(f"  ⛔ Empty         : {skipped_empty:>6,}")
print(f"\n  Per-category counts after cleaning:")
cat_counts = Counter(m['category'] for m in meta_list)
for cat, cnt in sorted(cat_counts.items()):
    print(f"    {cat:<42} {cnt:>5}")
print(f"\nSample cleaned post:")
print(cleaned_docs[0][:400])

Cleaning posts: 100%|██████████| 38748/38748 [00:11<00:00, 3518.16it/s]


── Cleaning Summary ────────────────────────────────────────
  Total loaded     : 38,748
  ✅ Kept          : 30,425
  ⛔ Too short     :  8,322  (< 50 words after cleaning)
  ⛔ Empty         :      1

  Per-category counts after cleaning:
    alt.atheism                                 1451
    comp.graphics                               1458
    comp.os.ms-windows.misc                     1422
    comp.sys.ibm.pc.hardware                    1583
    comp.sys.mac.hardware                       1509
    comp.windows.x                              1537
    misc.forsale                                1366
    rec.autos                                   1498
    rec.motorcycles                             1519
    rec.sport.baseball                          1430
    rec.sport.hockey                            1570
    sci.crypt                                   1596
    sci.electronics                             1566
    sci.med                                     1588
    sci.space      

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 9 — LOAD EMBEDDING MODEL ONTO GPU
#
# MODEL CHOICE: all-MiniLM-L6-v2
# ─────────────────────────────────────────────────────────────────────────
# Dimension  : 384  (vs 768 for mpnet — half storage, half VRAM, 2× faster)
# Speed      : ~14k sentences/sec on T4 → full corpus in ~6 min
# Quality    : Trained on 1B+ sentence pairs via contrastive learning;
#              strong semantic similarity for forum/news text
# Size       : ~90MB download, cached after first run
#
# Why NOT other models:
#   all-mpnet-base-v2      : better quality but 2× slower, 768-dim,
#                            overkill for this corpus size
#   text-embedding-ada-002 : best quality but needs OpenAI key + costs
#                            ~$0.27 for 18k posts — not "lightweight"
#   all-MiniLM-L12-v2      : marginally better than L6, 20% slower,
#                            not worth the trade-off here
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from sentence_transformers import SentenceTransformer
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading all-MiniLM-L6-v2 on {DEVICE.upper()}...")
print("(~90MB download on first run, cached after that)\n")

model    = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
test_emb = model.encode(["NASA space shuttle launch"], normalize_embeddings=True)

print(f"✅ Model loaded")
print(f"✅ Embedding dimension : {test_emb.shape[1]}")   # must be 384
print(f"✅ Running on         : {DEVICE.upper()}")

Loading all-MiniLM-L6-v2 on CUDA...
(~90MB download on first run, cached after that)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model loaded
✅ Embedding dimension : 384
✅ Running on         : CUDA


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 10 — GENERATE EMBEDDINGS + SAVE NUMPY BACKUP
#
# normalize_embeddings=True is MANDATORY.
#   ChromaDB collection uses cosine space which expects unit vectors.
#   Without normalization similarity scores are wrong.
#   Verify: all L2 norms should print as ~1.0000
#
# batch_size=64 is safe for free T4 (15GB VRAM).
#   MiniLM weights ~90MB + activations per batch ~50MB = well within limit.
#   Do not exceed 128 — longer posts can OOM at higher batch sizes.
#
# embeddings_backup.npy is saved to Drive immediately after encoding.
#   If Colab disconnects before ChromaDB insert finishes, load this
#   file and skip the 7-min re-embedding step entirely.
#   Parts 2 & 3 also load this directly for clustering operations.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import numpy as np
import time

print(f"Embedding {len(cleaned_docs):,} documents on {DEVICE.upper()}...")
print(f"Estimated time : ~6–8 min on T4  |  ~45–60 min on CPU\n")

t0 = time.time()

embeddings = model.encode(
    cleaned_docs,
    batch_size           = 64,
    show_progress_bar    = True,
    normalize_embeddings = True,   # unit vectors for cosine similarity
    convert_to_numpy     = True,   # return numpy not torch tensor
    device               = DEVICE
)

elapsed = time.time() - t0
print(f"\n✅ Embedding complete")
print(f"   Shape  : {embeddings.shape}")             # (N, 384)
print(f"   Time   : {elapsed/60:.1f} min")
print(f"   Speed  : {len(cleaned_docs)/elapsed:.0f} docs/sec")

# Sanity check — all norms must be ~1.0 to confirm normalization worked
norms = np.linalg.norm(embeddings[:100], axis=1)
print(f"   Norms  : min={norms.min():.4f}  max={norms.max():.4f}  (should be ~1.0)")

# Save numpy backup to Drive immediately
# Save numpy backup to Drive immediately
EMB_PATH = os.path.join(PERSIST_DIR, "embeddings_backup.npy")
np.save(EMB_PATH, embeddings)

# ── VERIFY the file actually wrote to Drive ────────────────────────────
# Previous sessions failed silently here — Drive appeared mounted but
# writes weren't flushing. These asserts catch that immediately.
assert os.path.exists(EMB_PATH), \
    "❌ embeddings_backup.npy failed to save — Drive may not be mounted"
size_mb = os.path.getsize(EMB_PATH) / 1e6
assert size_mb > 10, \
    f"❌ File too small ({size_mb:.1f} MB) — write was corrupted, re-run cell"

print(f"\n✅ Numpy backup → {EMB_PATH}")
print(f"   Size on disk : {size_mb:.1f} MB  (verified ✅)")

Embedding 30,425 documents on CUDA...
Estimated time : ~6–8 min on T4  |  ~45–60 min on CPU



Batches:   0%|          | 0/476 [00:00<?, ?it/s]


✅ Embedding complete
   Shape  : (30425, 384)
   Time   : 1.5 min
   Speed  : 338 docs/sec
   Norms  : min=1.0000  max=1.0000  (should be ~1.0)

✅ Numpy backup → /content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb/embeddings_backup.npy
   Size on disk : 46.7 MB  (verified ✅)


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 11 — CREATE CHROMADB COLLECTION
#
# VECTOR STORE CHOICE: ChromaDB PersistentClient
# ─────────────────────────────────────────────────────────────────────────
# Writes chroma.sqlite3 to PERSIST_DIR on your Drive.
# Parts 2 & 3 reconnect with:
#   chromadb.PersistentClient(path=PERSIST_DIR)
# No server process, no cloud account, no network dependency.
#
# hnsw:space="cosine" MUST match normalize_embeddings=True above.
#   Cosine distance on unit vectors = 1 - dot_product.
#   Results are returned as distances (0=identical, 2=opposite).
#   Convert to similarity with: similarity = 1 - distance
#
# Why NOT alternatives:
#   FAISS      : faster ANN, but no metadata filtering, no Drive
#                persistence without extra serialization code
#   Pinecone   : managed cloud, great API — needs account + internet
#                per query, violates "lightweight" constraint
#   Weaviate   : powerful but needs Docker or cloud, not Colab-friendly
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(
    path     = PERSIST_DIR,
    settings = Settings(anonymized_telemetry=False)
)

# Delete + recreate so this cell is safely re-runnable without duplicates
try:
    client.delete_collection("newsgroups_20ng")
    print("⚠️  Dropped existing collection (safe re-run)")
except Exception:
    pass

collection = client.create_collection(
    name     = "newsgroups_20ng",
    metadata = {"hnsw:space": "cosine"}
)

print(f"✅ Collection : {collection.name}")
print(f"✅ Space      : cosine")
print(f"✅ Saved to   : {PERSIST_DIR}/chroma.sqlite3")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Collection : newsgroups_20ng
✅ Space      : cosine
✅ Saved to   : /content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb/chroma.sqlite3


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 12 — INSERT ALL DOCUMENTS INTO CHROMADB
#
# Why batch size 500:
#   ChromaDB has an internal limit of ~41k items per call.
#   500 stays well within that and gives useful progress feedback.
#   Larger batches don't meaningfully speed up insertion.
#
# Why string IDs:
#   Integer IDs silently fail in chromadb 0.4.x — always use str(i).
#
# embeddings.tolist():
#   ChromaDB requires Python lists not numpy arrays.
#   .tolist() converts the full (N, 384) array at once.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import time
from tqdm import tqdm

def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

ids  = [str(i) for i in range(len(cleaned_docs))]   # must be strings
embs = embeddings.tolist()                            # numpy → Python list

t0             = time.time()
total_inserted = 0

for b_ids, b_docs, b_embs, b_meta in tqdm(
    zip(
        chunked(ids,          500),
        chunked(cleaned_docs, 500),
        chunked(embs,         500),
        chunked(meta_list,    500),
    ),
    total = (len(cleaned_docs) // 500) + 1,
    desc  = "Inserting into ChromaDB"
):
    collection.add(
        ids        = list(b_ids),
        documents  = list(b_docs),
        embeddings = list(b_embs),
        metadatas  = list(b_meta)
    )
    total_inserted += len(b_ids)

print(f"\n✅ Inserted  : {total_inserted:,} documents")
print(f"✅ Confirmed : {collection.count():,} in collection")
print(f"✅ Time      : {time.time()-t0:.1f}s")

Inserting into ChromaDB: 100%|██████████| 61/61 [03:12<00:00,  3.16s/it]


✅ Inserted  : 30,425 documents
✅ Confirmed : 30,425 in collection
✅ Time      : 192.5s


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 13 — SAVE MANIFEST.JSON
#
# This is the single file that Parts 2 & 3 import to reload everything.
# It contains all paths, model name, dimensions, and category mappings
# so Parts 2 & 3 never need to hardcode any paths themselves.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import json

manifest = {
    "persist_dir"       : PERSIST_DIR,
    "collection_name"   : "newsgroups_20ng",
    "embedding_model"   : "all-MiniLM-L6-v2",
    "embedding_dim"     : 384,
    "total_docs"        : collection.count(),
    "embeddings_backup" : EMB_PATH,
    "categories"        : unique_cats,
    "cat_to_idx"        : cat_to_idx,
}

MANIFEST_PATH = os.path.join(PERSIST_DIR, "manifest.json")
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

# ── VERIFY manifest actually wrote ────────────────────────────────────
assert os.path.exists(MANIFEST_PATH), \
    "❌ manifest.json failed to save — re-run this cell"

# Read it back and confirm it parses correctly
with open(MANIFEST_PATH) as f:
    verify = json.load(f)
assert verify["collection_name"] == "newsgroups_20ng", \
    "❌ manifest.json wrote but content is wrong — re-run this cell"
assert verify["total_docs"] > 0, \
    "❌ manifest.json shows 0 docs — ChromaDB insert may have failed"

print(f"✅ Manifest saved   → {MANIFEST_PATH}")
print(f"   Size             : {os.path.getsize(MANIFEST_PATH)/1e3:.1f} KB  (verified ✅)")
print(f"   Docs recorded    : {verify['total_docs']:,}")
print(f"   Categories       : {len(verify['categories'])}")
print()
print(json.dumps(manifest, indent=2))

✅ Manifest saved   → /content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb/manifest.json
   Size             : 1.4 KB  (verified ✅)
   Docs recorded    : 30,425
   Categories       : 20

{
  "persist_dir": "/content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb",
  "collection_name": "newsgroups_20ng",
  "embedding_model": "all-MiniLM-L6-v2",
  "embedding_dim": 384,
  "total_docs": 30425,
  "embeddings_backup": "/content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb/embeddings_backup.npy",
  "categories": [
    "alt.atheism",
    "comp.graphics",
    "comp.os.ms-windows.misc",
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.windows.x",
    "misc.forsale",
    "rec.autos",
    "rec.motorcycles",
    "rec.sport.baseball",
    "rec.sport.hockey",
    "sci.crypt",
    "sci.electronics",
    "sci.med",
    "sci.space",
    "soc.religion.christian",
    "talk.politics.guns",
    "talk.politics.mideast",
    "talk.politics.misc",
    "talk.religion.mi

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 14 — VERIFY: END-TO-END SEMANTIC SEARCH TEST
#
# Tests the full pipeline:
#   query string → embed → ChromaDB cosine search → ranked results
#
# category_filter demonstrates metadata filtering — the "filtered
# retrieval" requirement from the problem statement. Part 2 uses this
# to restrict cache lookups to specific topic areas.
#
# Expected behaviour:
#   "NASA space shuttle"     → results from sci.space
#   "Windows driver crash"   → results from comp.os.ms-windows.misc
#   "gun control"            → results from talk.politics.guns
#   filtered query           → ONLY results from specified category
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def semantic_search(query: str, n: int = 5, category_filter: str = None):
    """
    Embed query and retrieve top-n most similar documents from ChromaDB.
    similarity = 1 - cosine_distance  (1.0 = identical, 0.0 = unrelated)
    """
    q_emb = model.encode(
        [query],
        normalize_embeddings = True,
        device               = DEVICE
    ).tolist()

    where = {"category": category_filter} if category_filter else None

    res = collection.query(
        query_embeddings = q_emb,
        n_results        = n,
        where            = where,
        include          = ["documents", "metadatas", "distances"]
    )

    header = f"Query: '{query}'"
    if category_filter:
        header += f"  [filter: {category_filter}]"
    print(f"\n{header}")
    print("─" * 70)
    for doc, meta, dist in zip(
        res['documents'][0],
        res['metadatas'][0],
        res['distances'][0]
    ):
        sim = 1 - dist    # convert cosine distance → similarity score
        print(f"  [{meta['category']}]  sim={sim:.3f}  file={meta['filename']}")
        print(f"  {doc[:200]}...")
        print()

# Cross-category queries — should return correct categories
semantic_search("NASA space shuttle launch orbit mission")
semantic_search("Windows 95 graphics driver blue screen crash")
semantic_search("gun control second amendment rights debate")
semantic_search("atheism god religion existence proof")

# Filtered retrieval — should ONLY return from specified category
semantic_search("fast processor benchmark speed",
                category_filter="comp.sys.mac.hardware")
semantic_search("car engine repair torque",
                category_filter="rec.autos")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Query: 'NASA space shuttle launch orbit mission'
──────────────────────────────────────────────────────────────────────
  [sci.space]  sim=0.540  file=4672
  Space FAQ 09/15 - Mission Schedules. Archive-name: space/schedule Last-modified: $Date: 93/04/01 14:39:23 $ SPACE SHUTTLE ANSWERS, LAUNCH SCHEDULES, TV COVERAGE SHUTTLE LAUNCHINGS AND LANDINGS; SCHEDU...

  [sci.space]  sim=0.540  file=59904
  Space FAQ 09/15 - Mission Schedules. Archive-name: space/schedule Last-modified: $Date: 93/04/01 14:39:23 $ SPACE SHUTTLE ANSWERS, LAUNCH SCHEDULES, TV COVERAGE SHUTTLE LAUNCHINGS AND LANDINGS; SCHEDU...

  [sci.space]  sim=0.509  file=15151
  How can you see the launch of the Space Shuttle ?. Sorry for asking a question that's not entirely based on the technical aspects of space, but I couldn't find the answer on the FAQs ! I'm currently i...

  [sci.space]  sim=0.509  file=61199
  How can you see the launch of the Space Shuttle ?. Sorry for asking a question that's not entirely based on t

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 15 — FINAL SUMMARY + HARD VERIFICATION
# Does NOT let you proceed to Part 2 unless all 3 files are confirmed
# saved on Drive with correct sizes.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, torch, json

print("── Verifying all files saved to Drive ────────────────────────────────")

checks = {
    "embeddings_backup.npy" : 10.0,    # must be > 10 MB
    "chroma.sqlite3"        : 1.0,     # must be > 1 MB
    "manifest.json"         : 0.001,   # just needs to exist and parse
}

all_ok = True
for fname, min_mb in checks.items():
    fpath  = os.path.join(PERSIST_DIR, fname)
    exists = os.path.exists(fpath)
    size   = os.path.getsize(fpath) / 1e6 if exists else 0
    ok     = exists and size >= min_mb
    all_ok = all_ok and ok
    status = "✅" if ok else "❌"
    note   = "" if ok else f"  ← PROBLEM (expected >{min_mb}MB, got {size:.2f}MB)"
    print(f"  {status} {fname:<28} {size:>8.1f} MB{note}")

print()

if not all_ok:
    # Hard stop — tell exactly which cell to re-run
    print("❌ NOT ALL FILES SAVED — do NOT open Part 2 yet")
    print()
    print("   Fix:")
    if not os.path.exists(os.path.join(PERSIST_DIR, "embeddings_backup.npy")):
        print("   → Re-run Cell 10 (embedding)")
    if not os.path.exists(os.path.join(PERSIST_DIR, "chroma.sqlite3")):
        print("   → Re-run Cell 11 + Cell 12 (ChromaDB)")
    if not os.path.exists(os.path.join(PERSIST_DIR, "manifest.json")):
        print("   → Re-run Cell 13 (manifest)")
    raise RuntimeError("Part 1 incomplete — fix missing files before Part 2")

# All files confirmed — safe to proceed
print("✅ ALL 3 FILES CONFIRMED ON DRIVE")
print("   You will never need to re-run Part 1 again")
print()
print(f"── Stats ─────────────────────────────────────────────────────────────")
print(f"  Documents : {collection.count():,}")
print(f"  Embeddings: {embeddings.shape}")
print(f"  GPU used  : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print()
print(f"── Part 2 reload block ───────────────────────────────────────────────")
print(f"  Paste this at the top of your Part 2 notebook:\n")

RELOAD = f'''from google.colab import drive
drive.mount('/content/drive')

import json, numpy as np, chromadb, torch, logging
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

PERSIST_DIR = "{PERSIST_DIR}"

with open(f"{{PERSIST_DIR}}/manifest.json") as f:
    manifest = json.load(f)

client     = chromadb.PersistentClient(
                 path=PERSIST_DIR,
                 settings=Settings(anonymized_telemetry=False))
collection = client.get_collection(manifest["collection_name"])

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
model      = SentenceTransformer(manifest["embedding_model"], device=DEVICE)
embeddings = np.load(manifest["embeddings_backup"])

print(f"✅ Collection : {{collection.count():,}} docs")
print(f"✅ Embeddings : {{embeddings.shape}}")
print(f"✅ Device     : {{DEVICE.upper()}}")'''

print(RELOAD)

── Verifying all files saved to Drive ────────────────────────────────
  ✅ embeddings_backup.npy            46.7 MB
  ✅ chroma.sqlite3                  384.1 MB
  ✅ manifest.json                     0.0 MB

✅ ALL 3 FILES CONFIRMED ON DRIVE
   You will never need to re-run Part 1 again

── Stats ─────────────────────────────────────────────────────────────
  Documents : 30,425
  Embeddings: (30425, 384)
  GPU used  : Tesla T4

── Part 2 reload block ───────────────────────────────────────────────
  Paste this at the top of your Part 2 notebook:

from google.colab import drive
drive.mount('/content/drive')

import json, numpy as np, chromadb, torch, logging
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

logging.getLogger("chromadb.telemetry.product.posthog").setLevel(logging.CRITICAL)

PERSIST_DIR = "/content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb"

with open(f"{PERSIST_DIR}/manifest.json") as f:
    manifest = json.load(f)

clien

In [ ]:
# OPTIONAL CLEANUP — removes duplicate docs, keeps exactly one copy
# Run this once before starting Part 2
# Takes ~3 min

import chromadb, json
from chromadb.config import Settings

PERSIST_DIR = "/content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb"

client = chromadb.PersistentClient(
            path=PERSIST_DIR,
            settings=Settings(anonymized_telemetry=False))

collection = client.get_collection("newsgroups_20ng")

print(f"Before : {collection.count():,} docs")


# Get all documents and metadata
result = collection.get(limit=collection.count(), include=["documents","metadatas"])

docs = result["documents"]
ids  = result["ids"]


# Find duplicate documents using hashing
import hashlib

seen     = set()
keep_ids = []
drop_ids = []

for doc_id, doc in zip(ids, docs):

    # Use first 200 characters as fingerprint
    fingerprint = hashlib.md5(doc[:200].encode()).hexdigest()

    if fingerprint in seen:
        drop_ids.append(doc_id)
    else:
        seen.add(fingerprint)
        keep_ids.append(doc_id)


print(f"Keep  : {len(keep_ids):,}")
print(f"Drop  : {len(drop_ids):,} duplicates")


# Delete duplicates in batches
from tqdm import tqdm

BATCH = 500

for start in tqdm(range(0, len(drop_ids), BATCH), desc="Removing duplicates"):
    collection.delete(ids=drop_ids[start:start+BATCH])


print(f"\nAfter  : {collection.count():,} docs")
print("Duplicates removed — ready for Part 2")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Before : 30,425 docs


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Keep  : 15,176
Drop  : 15,249 duplicates


Removing duplicates: 100%|██████████| 31/31 [00:10<00:00,  2.90it/s]


After  : 15,176 docs
Duplicates removed — ready for Part 2


In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 15 — FINAL SUMMARY + HOW TO USE IN PARTS 2 & 3
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, torch

print("✅ PART 1 COMPLETE\n")
print("── Files saved to Drive ──────────────────────────────────────────────")

for fname, purpose in {
    "chroma.sqlite3"        : "Vector index  — ChromaDB queries in Parts 2 & 3",
    "embeddings_backup.npy" : "Raw embeddings — numpy array for clustering/cache",
    "manifest.json"         : "Config file   — single import for Parts 2 & 3",
}.items():
    fpath = os.path.join(PERSIST_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f"  ✅ {fname:<28} {size:>8.1f} MB  │  {purpose}")
    else:
        print(f"  ❌ {fname:<28} NOT FOUND — re-run the relevant cell")

print(f"""
── Drive location ─────────────────────────────────────────────────────────
  {PERSIST_DIR}/
  ├── chroma.sqlite3
  ├── embeddings_backup.npy
  └── manifest.json

── Stats ──────────────────────────────────────────────────────────────────
  Documents indexed : {collection.count():,}
  Embedding model   : all-MiniLM-L6-v2  (384-dim, cosine space)
  GPU used          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}

── HOW TO RELOAD IN PART 2 & PART 3 ──────────────────────────────────────
  Copy-paste this block at the top of your next notebook:

  ┌─────────────────────────────────────────────────────────────────────┐
  │ from google.colab import drive                                      │
  │ drive.mount('/content/drive')                                       │
  │                                                                     │
  │ import json, numpy as np, chromadb, torch                          │
  │ from chromadb.config import Settings                                │
  │ from sentence_transformers import SentenceTransformer               │
  │                                                                     │
  │ PERSIST_DIR = "{PERSIST_DIR}"
  │                                                                     │
  │ with open(f"{{PERSIST_DIR}}/manifest.json") as f:                  │
  │     manifest = json.load(f)                                         │
  │                                                                     │
  │ client = chromadb.PersistentClient(                                 │
  │     path=PERSIST_DIR,                                               │
  │     settings=Settings(anonymized_telemetry=False))                  │
  │ collection = client.get_collection(manifest["collection_name"])     │
  │                                                                     │
  │ DEVICE = "cuda" if torch.cuda.is_available() else "cpu"            │
  │ model  = SentenceTransformer(                                       │
  │     manifest["embedding_model"], device=DEVICE)                     │
  │                                                                     │
  │ embeddings = np.load(manifest["embeddings_backup"])                 │
  │                                                                     │
  │ print(f"Collection : {{collection.count():,}} docs")               │
  │ print(f"Embeddings : {{embeddings.shape}}")                        │
  └─────────────────────────────────────────────────────────────────────┘
""")

✅ PART 1 COMPLETE

── Files saved to Drive ──────────────────────────────────────────────
  ✅ chroma.sqlite3                  190.1 MB  │  Vector index  — ChromaDB queries in Parts 2 & 3
  ✅ embeddings_backup.npy            22.7 MB  │  Raw embeddings — numpy array for clustering/cache
  ✅ manifest.json                     0.0 MB  │  Config file   — single import for Parts 2 & 3

── Drive location ─────────────────────────────────────────────────────────
  /content/drive/MyDrive/twenty+newsgroups/newsgroups_chromadb/
  ├── chroma.sqlite3
  ├── embeddings_backup.npy
  └── manifest.json

── Stats ──────────────────────────────────────────────────────────────────
  Documents indexed : 14,792
  Embedding model   : all-MiniLM-L6-v2  (384-dim, cosine space)
  GPU used          : Tesla T4

── HOW TO RELOAD IN PART 2 & PART 3 ──────────────────────────────────────
  Copy-paste this block at the top of your next notebook:

  ┌─────────────────────────────────────────────────────────────────────┐